# Part II - Temporal K-means (variant 3)

In [ ]:
# 0. Imports and paths
import os
import os.path as op
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib

from nilearn import image, plotting, datasets
from nilearn.maskers import NiftiMasker
from nilearn.glm.first_level import make_first_level_design_matrix
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

DATA_ROOT = "."

FMRI = op.join(DATA_ROOT, "fMRI/derivatives/preprocessed_data/gaussian_smoothing/tfMRI_LANGUAGE_LR_smoothed-6mm.nii.gz")
EVENTS = op.join(DATA_ROOT, "fMRI/tfMRI_LANGUAGE_LR/tfMRI_LANGUAGE_LR_task_onsets.csv")
MOTION = op.join(DATA_ROOT, "fMRI/derivatives/preprocessed_data/motion_correction/tfMRI_LANGUAGE_LR_moco.nii.gz.par")
ANAT = op.join(DATA_ROOT, "T1/derivatives/preprocessed_data/registration/T1w_mni.nii.gz")

RESULTS = "results/"
os.makedirs(RESULTS, exist_ok=True)

# ---- Parameters -------------------------------------------------------------
TR = 0.72                 # HCP language task
N_REMOVED = 10            # volumes removed during preprocessing (Part I)
HIGH_PASS = 0.008         # Hz (well below the ~60 s story/math cycle)
K_RANGE = range(2, 16)    # candidate numbers of clusters
K = 4                     # chosen number of clusters (see model selection below)
THR_FRAC = 0.30           # maps thresholded at 30% of their max |value|
RNG = 0

In [ ]:
# 1. Load data, build brain mask, preprocess voxel time series
fmri_img = image.load_img(FMRI)
n_scans = fmri_img.shape[-1]
print("fMRI shape:", fmri_img.shape)

# MNI152 brain mask resampled to the fMRI grid
mni_mask = datasets.load_mni152_brain_mask(resolution=2)
mask_img = image.resample_to_img(mni_mask, image.index_img(fmri_img, 0),
                                 interpolation="nearest")

# detrend + high-pass filter + z-score each voxel, restricted to the mask
masker = NiftiMasker(mask_img=mask_img, detrend=True, high_pass=HIGH_PASS,
                     t_r=TR, standardize="zscore_sample")
X = masker.fit_transform(fmri_img)          # (n_timepoints, n_voxels)
X = np.nan_to_num(X)
n_t, n_vox = X.shape
print(f"Data matrix: {n_t} time points x {n_vox} voxels")

In [ ]:
# 2. SVD projection (K-means on Z = U S gives the same partition as on X)
U, S, Vt = np.linalg.svd(X, full_matrices=False)
Z = U * S                                   # (n_t, n_t): distances between time points preserved
assert np.allclose(np.linalg.norm(X[0] - X[1]), np.linalg.norm(Z[0] - Z[1]))

In [ ]:
# 3. Choice of the number of clusters (elbow + silhouette)
inertias, silhouettes = [], []
for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=50, random_state=RNG).fit(Z)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(Z, km.labels_))
    print(f"k={k:2d}  inertia={km.inertia_:.1f}  silhouette={silhouettes[-1]:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(list(K_RANGE), inertias, "ko-")
ax[0].set(xlabel="Number of clusters k", ylabel="Inertia (WCSS)", title="Elbow")
ax[1].plot(list(K_RANGE), silhouettes, "ko-")
ax[1].axvline(K, color="r", ls="--", label=f"chosen k = {K}")
ax[1].set(xlabel="Number of clusters k", ylabel="Silhouette score", title="Silhouette")
ax[1].legend()
for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(op.join(RESULTS, "kmeans_choice_of_k.png"), dpi=200)
plt.show()
print("k maximising the silhouette:", list(K_RANGE)[int(np.argmax(silhouettes))])

In [ ]:
# 4. Final K-means
kmeans = KMeans(n_clusters=K, n_init=100, random_state=RNG).fit(Z)
labels = kmeans.labels_
centroids = kmeans.cluster_centers_ @ Vt    # back to voxel space, (K, n_vox)

# sanity check: centroid = mean z-scored volume of its time points
for c in range(K):
    assert np.allclose(centroids[c], X[labels == c].mean(axis=0), atol=1e-4)
print("Cluster sizes:", {c + 1: int((labels == c).sum()) for c in range(K)})

In [ ]:
# 5. Centroid maps overlaid on the MNI T1, axial view only
cut_z = np.linspace(-30, 60, 8)
for c in range(K):
    cimg = masker.inverse_transform(centroids[c])
    vmax = np.abs(centroids[c]).max()
    plotting.plot_stat_map(cimg, bg_img=ANAT, display_mode="z", cut_coords=cut_z,
                           threshold=THR_FRAC * vmax, vmax=vmax, colorbar=True,
                           title=f"Cluster {c + 1} centroid (n = {(labels == c).sum()} time points)",
                           output_file=op.join(RESULTS, f"centroid_cluster{c + 1}.png"))
    plt.show()

In [ ]:
# 6. Atlases (Yeo 7 networks + Harvard-Oxford) to name the regions
# --- Yeo 2011, 7 networks
yeo = datasets.fetch_atlas_yeo_2011()
yeo_img = image.load_img(yeo["thick_7"])
YEO_NAMES = ["Visual", "Somatomotor", "Dorsal attention", "Ventral attention",
             "Limbic", "Fronto-parietal", "Default mode"]

# --- Harvard-Oxford cortical atlas
ho = datasets.fetch_atlas_harvard_oxford("cort-maxprob-thr25-2mm")
ho_img = image.load_img(ho.maps)
ho_labels = list(ho.labels)

# reference plots of the atlases
plotting.plot_roi(yeo_img, display_mode="z", cut_coords=cut_z, cmap="Paired",
                  title="Yeo 7 networks", output_file=op.join(RESULTS, "atlas_yeo7.png"))
plotting.plot_roi(ho_img, display_mode="z", cut_coords=cut_z,
                  title="Harvard-Oxford cortical atlas", output_file=op.join(RESULTS, "atlas_ho.png"))
plt.show()

# atlas labels expressed in the same voxel space as X (no standardisation applied)
from nilearn.masking import apply_mask
def atlas_in_mask(atlas_img):
    res = image.resample_to_img(atlas_img, masker.mask_img_, interpolation="nearest")
    return np.round(apply_mask(res, masker.mask_img_)).astype(int)

yeo_vox = atlas_in_mask(yeo_img)            # 0 = outside the 7 networks, 1..7
ho_vox = atlas_in_mask(ho_img)              # 0 = background, 1..48

In [ ]:
# 7. Which networks / regions does each centroid recruit?
# 7a. Proportion of supra-threshold voxels falling in each Yeo network
rows = []
for c in range(K):
    thr = THR_FRAC * np.abs(centroids[c]).max()
    for sign, name in [(+1, "positive"), (-1, "negative")]:
        active = (sign * centroids[c]) > thr
        n_act = max(active.sum(), 1)
        row = {"cluster": c + 1, "sign": name, "n_voxels": int(active.sum())}
        for i, net in enumerate(YEO_NAMES, start=1):
            row[net] = 100 * (active & (yeo_vox == i)).sum() / n_act
        row["outside Yeo"] = 100 * (active & (yeo_vox == 0)).sum() / n_act
        rows.append(row)
yeo_table = pd.DataFrame(rows).round(1)
print(yeo_table.to_string(index=False))
yeo_table.to_csv(op.join(RESULTS, "yeo_overlap.csv"), index=False)

fig, axes = plt.subplots(1, K, figsize=(4 * K, 4), sharey=True)
for c in range(K):
    sub = yeo_table[yeo_table.cluster == c + 1].set_index("sign")[YEO_NAMES]
    x = np.arange(len(YEO_NAMES))
    axes[c].bar(x - 0.2, sub.loc["positive"], 0.4, label="positive", color="tab:red")
    axes[c].bar(x + 0.2, sub.loc["negative"], 0.4, label="negative", color="tab:blue")
    axes[c].set_xticks(x)
    axes[c].set_xticklabels(YEO_NAMES, rotation=70, ha="right", fontsize=8)
    axes[c].set_title(f"Cluster {c + 1}")
axes[0].set_ylabel("% of supra-threshold voxels")
axes[0].legend()
plt.tight_layout()
plt.savefig(op.join(RESULTS, "yeo_overlap.png"), dpi=200)
plt.show()

# 7b. Peaks of each centroid labelled with the Harvard-Oxford atlas
from nilearn.reporting import get_clusters_table
ho_data = ho_img.get_fdata().astype(int)
inv_aff = np.linalg.inv(ho_img.affine)

def ho_name(xyz):
    ijk = np.round(nib.affines.apply_affine(inv_aff, xyz)).astype(int)
    if np.any(ijk < 0) or np.any(ijk >= ho_data.shape):
        return "n/a"
    return ho_labels[ho_data[tuple(ijk)]]

peak_tables = []
for c in range(K):
    cimg = masker.inverse_transform(centroids[c])
    thr = THR_FRAC * np.abs(centroids[c]).max()
    tab = get_clusters_table(cimg, stat_threshold=thr, cluster_threshold=20, two_sided=True)
    tab["HO region"] = [ho_name([r.X, r.Y, r.Z]) for r in tab.itertuples()]
    tab.insert(0, "cluster", c + 1)
    peak_tables.append(tab)
    print(f"\n=== Cluster {c + 1} ===")
    print(tab[["Cluster ID", "X", "Y", "Z", "Peak Stat", "Cluster Size (mm3)", "HO region"]].head(10).to_string(index=False))
pd.concat(peak_tables).to_csv(op.join(RESULTS, "centroid_peaks_HO.csv"), index=False)

In [ ]:
# 8. Motion (framewise displacement) aligned with the preprocessed run
par = np.loadtxt(MOTION)                    # MCFLIRT: 3 rotations (rad) + 3 translations (mm)
if par.shape[0] == n_scans + N_REMOVED:
    par = par[N_REMOVED:]
assert par.shape[0] == n_scans, f"motion parameters ({par.shape[0]}) vs volumes ({n_scans})"
d = np.vstack([np.zeros((1, 6)), np.abs(np.diff(par, axis=0))])
fd = d[:, 3:].sum(1) + 50.0 * d[:, :3].sum(1)    # Power et al., 50 mm sphere

In [ ]:
# 9. GLM regressors of Part I (blocks + events), HRF-convolved
ev = pd.read_csv(EVENTS)
print(ev.head(12), "\n", ev.columns.tolist())

# >>> ADAPT these names to the real columns / labels of the onsets file <<<
COL_ONSET, COL_DUR, COL_TYPE = "onset", "duration", "trial_type"
ev = ev.rename(columns={COL_ONSET: "onset", COL_DUR: "duration", COL_TYPE: "trial_type"})
ev["trial_type"] = ev["trial_type"].astype(str).str.lower()

# shift onsets because the first N_REMOVED volumes were removed
ev["onset"] = ev["onset"] - N_REMOVED * TR
end = ev["onset"] + ev["duration"]
ev = ev[end > 0].copy()
ev["duration"] = np.where(ev["onset"] < 0, end[ev.index], ev["duration"])
ev["onset"] = ev["onset"].clip(lower=0)

# block regressors: use rows named 'story'/'math' if present, otherwise build
# them from the events of each task (new block when the gap exceeds GAP s)
GAP = 10.0
if {"story", "math"}.issubset(set(ev["trial_type"])):
    blocks = ev[ev.trial_type.isin(["story", "math"])]
    events_only = ev[~ev.trial_type.isin(["story", "math"])]
else:
    blk = []
    for task in ["story", "math"]:
        sub = ev[ev.trial_type.str.contains(task)].sort_values("onset")
        start = prev_end = None
        for r in sub.itertuples():
            if start is None:
                start = r.onset
            elif r.onset - prev_end > GAP:
                blk.append((start, prev_end - start, task))
                start = r.onset
            prev_end = r.onset + r.duration
        if start is not None:
            blk.append((start, prev_end - start, task))
    blocks = pd.DataFrame(blk, columns=["onset", "duration", "trial_type"])
    events_only = ev
all_ev = pd.concat([blocks, events_only])[["onset", "duration", "trial_type"]]

frame_times = np.arange(n_scans) * TR
dm = make_first_level_design_matrix(frame_times, all_ev, hrf_model="spm", drift_model=None)
dm = dm.drop(columns="constant")
dm["FD"] = fd
print("Regressors:", dm.columns.tolist())

In [ ]:
# 10. Binary cluster time courses and correlation with the regressors
binary = pd.DataFrame({f"Cluster {c + 1}": (labels == c).astype(float)
                       for c in range(min(K, 10))})
corr = pd.DataFrame([[np.corrcoef(binary[b], dm[r])[0, 1] for r in dm.columns]
                     for b in binary.columns], index=binary.columns, columns=dm.columns)
print(corr.round(2).to_string())
corr.to_csv(op.join(RESULTS, "cluster_glm_correlation.csv"))

fig, ax = plt.subplots(figsize=(1.0 * dm.shape[1] + 2, 0.9 * binary.shape[1] + 2))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(corr.shape[1]))
ax.set_xticklabels(corr.columns, rotation=60, ha="right")
ax.set_yticks(range(corr.shape[0]))
ax.set_yticklabels(corr.index)
for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, label="Pearson r")
ax.set_title("Cluster time courses vs. GLM regressors")
plt.tight_layout()
plt.savefig(op.join(RESULTS, "cluster_glm_correlation.png"), dpi=200)
plt.show()

In [ ]:
# 11. (Optional) cluster assignment over time vs. task blocks
fig, ax = plt.subplots(figsize=(12, 3))
for c in range(K):
    ax.scatter(frame_times[labels == c], np.full((labels == c).sum(), c + 1), s=12, label=f"Cluster {c + 1}")
for name, col in [("story", "tab:green"), ("math", "tab:orange")]:
    if name in dm.columns:
        ax.fill_between(frame_times, 0, dm[name] / dm[name].max() * (K + 1), color=col, alpha=0.2, label=f"{name} (block)")
ax.set(xlabel="Time (s)", ylabel="Cluster", yticks=range(1, K + 1))
ax.legend(ncol=3, fontsize=8)
plt.tight_layout()
plt.savefig(op.join(RESULTS, "cluster_timeline.png"), dpi=200)
plt.show()